# BloomBERT — Focal Loss Only

Single-intervention experiment: replace class-weighted cross-entropy with class-weighted **focal loss (γ=2)**, keep everything else identical to the baseline.

## Why focal loss
Plain cross-entropy treats all examples equally once weighted. Focal loss adds a $(1-p_t)^\gamma$ modulator that **down-weights examples the model already classifies well** (high $p_t$) and **up-weights uncertain examples** (low $p_t$). For our task this concentrates gradient on the easy↔medium and medium↔hard boundary cases, which is where the baseline's confusion matrix showed the biggest errors (294 easy→medium, 132 medium→easy, 63 hard→medium).

Reference: Lin et al. 2017, *Focal Loss for Dense Object Detection*.

## What's different from baseline
| Setting | Baseline | This notebook |
|---|---|---|
| Loss | class-weighted CE | **class-weighted focal (γ=2)** |
| Epochs | 4 | **8** |
| Unfreeze top layers | 3 | 3 |
| LR / LLRD / schedule / pooling / dropout / etc. | identical |

## Setup
1. Attach the `baylearn-bloom-data` Kaggle dataset.
2. Accelerator → GPU T4 x1.
3. Run All. ~35 min on T4 (longer than baseline because 8 epochs vs 4).
4. Output: `/kaggle/working/bloom_distilbert_focal/`. Includes all report artifacts.

In [ ]:
# ---- Cell 1: Imports + Config -------------------------------------------
import os, json, random, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Config = {
    'model_name'   : 'distilbert-base-uncased',
    'max_len'      : 128,
    'batch_size'   : 32,
    'epochs'       : 8,                # ↑ from baseline 4
    'lr'           : 2e-5,
    'llrd_decay'   : 0.9,
    'weight_decay' : 0.01,
    'warmup_frac'  : 0.10,
    'unfreeze_top' : 3,                # best from baseline ablation
    'num_classes'  : 3,
    'use_amp'      : True,
    'focal_gamma'  : 2.0,              # focal loss focusing parameter
}
LABEL_TO_ID = {'easy': 0, 'medium': 1, 'hard': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

DATA_DIR = '/kaggle/input/datasets/manarabdelshafy/baylearn-bloom-data'
OUT_DIR  = '/kaggle/working/bloom_distilbert_focal'
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'Focal gamma: {Config["focal_gamma"]}   Epochs: {Config["epochs"]}')

In [ ]:
# ---- Cell 2: Data (identical to baseline) -------------------------------
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.dropna(subset=['question', 'level'], inplace=True)
    df['level'] = df['level'].str.lower().str.strip()
    df.drop(df[~df['level'].isin(LABEL_TO_ID.keys())].index, inplace=True)
    print(f'  {name}: {len(df)}/{before}')

print('\nLabel distribution in train:')
print(train_df['level'].value_counts())

In [ ]:
# ---- Cell 3: Dataset + loaders (identical to baseline) ------------------
tokenizer = DistilBertTokenizerFast.from_pretrained(Config['model_name'])

class BloomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df['question'].astype(str).tolist()
        self.labels = [LABEL_TO_ID[l] for l in df['level']]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True,
                             padding='max_length', max_length=self.max_len,
                             return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = BloomDataset(train_df, tokenizer, Config['max_len'])
val_ds   = BloomDataset(val_df,   tokenizer, Config['max_len'])
test_ds  = BloomDataset(test_df,  tokenizer, Config['max_len'])

train_loader = DataLoader(train_ds, batch_size=Config['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ---- Cell 4: Build model (same as baseline — DistilBertForSequenceClassification + frozen-layer trick)
def build_model(unfreeze_top: int):
    model = DistilBertForSequenceClassification.from_pretrained(
        Config['model_name'], num_labels=Config['num_classes']
    )
    for p in model.parameters():
        p.requires_grad = False
    layers = model.distilbert.transformer.layer
    for i in range(6 - unfreeze_top, 6):
        for p in layers[i].parameters():
            p.requires_grad = True
    for p in model.pre_classifier.parameters(): p.requires_grad = True
    for p in model.classifier.parameters():     p.requires_grad = True
    return model.to(device)

_m = build_model(Config['unfreeze_top'])
total   = sum(p.numel() for p in _m.parameters())
train_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total: {total:,}   Trainable: {train_p:,}   ({100*train_p/total:.1f}%)')
del _m; torch.cuda.empty_cache()

In [ ]:
# ---- Cell 5: Class weights + Focal loss --------------------------------
counts = train_df['level'].value_counts()
tot = counts.sum()
weights = {l: tot / (Config['num_classes'] * counts[l]) for l in counts.index}
class_weights = torch.tensor(
    [weights[ID_TO_LABEL[i]] for i in range(Config['num_classes'])],
    dtype=torch.float, device=device,
)
print('Class weights:', {ID_TO_LABEL[i]: round(class_weights[i].item(), 3) for i in range(3)})

class FocalLoss(nn.Module):
    """Class-weighted focal loss (Lin et al. 2017).

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    where p_t is the predicted probability of the TRUE class. alpha_t is
    the per-class weight (inverse-frequency). When gamma=0 this reduces to
    plain class-weighted cross-entropy (baseline).
    """
    def __init__(self, gamma: float, alpha: torch.Tensor):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha   # (K,)

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)              # (B, K)
        log_pt    = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        pt        = log_pt.exp()                                # (B,)
        focal     = (1.0 - pt).clamp(min=1e-9).pow(self.gamma)  # (B,)
        alpha_t   = self.alpha[targets]                         # (B,)
        loss = -alpha_t * focal * log_pt                        # (B,)
        return loss.mean()

In [ ]:
# ---- Cell 6: LLRD parameter groups (identical to baseline) -------------
def build_llrd_parameter_groups(model, base_lr: float, decay: float, weight_decay: float):
    no_decay_names = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    groups = []
    for layer_idx in range(6):
        depth_from_top = 5 - layer_idx
        lr = base_lr * (decay ** depth_from_top)
        prefix = f'distilbert.transformer.layer.{layer_idx}.'
        decay_params, nodecay_params = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad or not n.startswith(prefix):
                continue
            (nodecay_params if any(nd in n for nd in no_decay_names) else decay_params).append(p)
        if decay_params:
            groups.append({'params': decay_params, 'lr': lr, 'weight_decay': weight_decay})
        if nodecay_params:
            groups.append({'params': nodecay_params, 'lr': lr, 'weight_decay': 0.0})
    head_decay, head_nodecay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'classifier' not in n and 'pre_classifier' not in n: continue
        (head_nodecay if any(nd in n for nd in no_decay_names) else head_decay).append(p)
    if head_decay:
        groups.append({'params': head_decay, 'lr': base_lr, 'weight_decay': weight_decay})
    if head_nodecay:
        groups.append({'params': head_nodecay, 'lr': base_lr, 'weight_decay': 0.0})
    return groups

In [ ]:
# ---- Cell 7: Train + eval loops (same as baseline) ----------------------
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        ids  = batch['input_ids'].to(device, non_blocking=True)
        attn = batch['attention_mask'].to(device, non_blocking=True)
        lab  = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast():
                out  = model(input_ids=ids, attention_mask=attn)
                loss = loss_fn(out.logits, lab)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out  = model(input_ids=ids, attention_mask=attn)
            loss = loss_fn(out.logits, lab)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0; preds, gold = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device, non_blocking=True)
        attn = batch['attention_mask'].to(device, non_blocking=True)
        lab  = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']:
            with autocast():
                out = model(input_ids=ids, attention_mask=attn)
        else:
            out = model(input_ids=ids, attention_mask=attn)
        total_loss += loss_fn(out.logits.float(), lab).item()
        preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
        gold.extend(lab.cpu().tolist())
    return total_loss / len(loader), f1_score(gold, preds, average='macro'), preds, gold

In [ ]:
# ---- Cell 8: Artifact-saving helper -------------------------------------
def save_test_artifacts(out_dir, preds, gold, test_loss, test_f1, history,
                        config, unfreeze_top, df_for_questions=None, prefix=''):
    target_names = ['easy', 'medium', 'hard']
    os.makedirs(out_dir, exist_ok=True)
    with open(f'{out_dir}/{prefix}training_history.json', 'w') as f:
        json.dump({'history': history, 'test_macro_f1': test_f1, 'test_loss': test_loss,
                   'config': config, 'unfreeze_top': unfreeze_top}, f, indent=2)
    text_report = classification_report(gold, preds, target_names=target_names, digits=4)
    with open(f'{out_dir}/{prefix}test_classification_report.txt', 'w') as f:
        f.write(f'TEST macro_f1 = {test_f1:.4f}   loss = {test_loss:.4f}\n\n')
        f.write(text_report)
    with open(f'{out_dir}/{prefix}test_classification_report.json', 'w') as f:
        json.dump(classification_report(gold, preds, target_names=target_names,
                                        digits=4, output_dict=True), f, indent=2)
    cm = confusion_matrix(gold, preds)
    with open(f'{out_dir}/{prefix}test_confusion_matrix.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['true \\ pred'] + target_names)
        for i, row in enumerate(cm):
            w.writerow([target_names[i]] + list(row))
    with open(f'{out_dir}/{prefix}test_predictions.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['idx', 'question', 'true_level', 'predicted_level', 'correct', 'source'])
        qs = df_for_questions['question'].tolist() if df_for_questions is not None else [''] * len(gold)
        src = df_for_questions['source'].tolist() if (df_for_questions is not None and 'source' in df_for_questions.columns) else [''] * len(gold)
        for i, (q, t, p) in enumerate(zip(qs, gold, preds)):
            w.writerow([i, q, target_names[t], target_names[p],
                        'yes' if t == p else 'no',
                        src[i] if i < len(src) else ''])
    with open(f'{out_dir}/{prefix}label_map.json', 'w') as f:
        json.dump(LABEL_TO_ID, f, indent=2)

In [ ]:
# ---- Cell 9: MAIN training run (focal loss, 8 epochs) -------------------
model = build_model(Config['unfreeze_top'])
loss_fn = FocalLoss(gamma=Config['focal_gamma'], alpha=class_weights)
optimizer = torch.optim.AdamW(
    build_llrd_parameter_groups(model, Config['lr'], Config['llrd_decay'], Config['weight_decay'])
)
total_steps  = len(train_loader) * Config['epochs']
warmup_steps = int(total_steps * Config['warmup_frac'])
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)
scaler = GradScaler(enabled=Config['use_amp'])

history = []
best_f1, best_state = -1.0, None
for epoch in range(1, Config['epochs'] + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, loss_fn)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': val_loss, 'val_macro_f1': val_f1})
    print(f'  epoch {epoch}/{Config["epochs"]}   train={tr_loss:.4f}  val={val_loss:.4f}  val_f1={val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
test_loss, test_f1, preds, gold = evaluate(model, test_loader, loss_fn)
print(f'\nTEST  macro_f1={test_f1:.4f}   loss={test_loss:.4f}')
print(classification_report(gold, preds, target_names=['easy','medium','hard'], digits=4))
print('Confusion matrix (rows=true, cols=pred):  easy / medium / hard')
print(confusion_matrix(gold, preds))

model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
save_test_artifacts(OUT_DIR, preds, gold, test_loss, test_f1, history,
                    Config, Config['unfreeze_top'], df_for_questions=test_df)
print(f'\nSaved to {OUT_DIR}')
print(f'\n=== FOCAL vs BASELINE ===')
print(f'Baseline (CE,    4 epochs, 3 layers):  test macro-F1 = 0.7261')
print(f'Focal    (γ=2.0, 8 epochs, 3 layers):  test macro-F1 = {test_f1:.4f}')
print(f'Delta:                                  {test_f1 - 0.7261:+.4f}')

In [ ]:
# ---- Cell 10: Sanity check on saved model -------------------------------
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
loaded_model = DistilBertForSequenceClassification.from_pretrained(OUT_DIR).to(device).eval()
loaded_tok = DistilBertTokenizerFast.from_pretrained(OUT_DIR)

samples = [
    'Define a semaphore.',
    'Compute the number of page faults for the reference string 1,2,3,4,1,2,5 under LRU with 3 frames.',
    'Design a deadlock-free dining philosophers protocol using only binary semaphores and justify why it avoids circular wait.',
]
with torch.no_grad():
    enc = loaded_tok(samples, padding=True, truncation=True,
                     max_length=Config['max_len'], return_tensors='pt').to(device)
    logits = loaded_model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = logits.argmax(dim=-1).cpu().tolist()
for s, p, pr in zip(samples, preds, probs):
    print(f'\n[{ID_TO_LABEL[p]}]  (easy={pr[0]:.2f}  medium={pr[1]:.2f}  hard={pr[2]:.2f})')
    print(f'  Q: {s}')

In [ ]:
# ---- Final cell: ZIP the output for reliable single-file download -------
import shutil, os
zip_path = shutil.make_archive('/kaggle/working/bloom_distilbert_focal', 'zip', OUT_DIR)
sz_mb = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({sz_mb:.1f} MB)')
print(f'\nDownload bloom_distilbert_focal.zip from the right-side Output panel,')
print(f'then unzip locally to question-generation-module/models/bloom_distilbert/')
print(f'(use a different folder if you want to keep multiple runs side-by-side).')